In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)


# ---------------------------------------------------------
# 1. Load dataset
# ---------------------------------------------------------

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="diagnosis")

print("=" * 70)
print("BREAST CANCER DATASET")
print("=" * 70)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", sorted(y.unique().tolist()))


# ---------------------------------------------------------
# 2. Train-test split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data :", X_test.shape)


# ---------------------------------------------------------
# 3. Feature scaling
# ---------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ---------------------------------------------------------
# 4. Define models
# ---------------------------------------------------------

models = {
    "logistic_regression": LogisticRegression(
        max_iter=10000,
        random_state=42
    ),

    "decision_tree": DecisionTreeClassifier(
        random_state=42
    ),

    "knn": KNeighborsClassifier(
        n_neighbors=5
    ),

    "naive_bayes": GaussianNB(),

    "random_forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}


# ---------------------------------------------------------
# 5. Create model folder
# ---------------------------------------------------------

os.makedirs("model", exist_ok=True)


# ---------------------------------------------------------
# 6. Train and evaluate models
# ---------------------------------------------------------

results = []

for name, model in models.items():

    # Logistic Regression and KNN use scaled data.
    if name in ["logistic_regression", "knn"]:
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]

        train_pred = model.predict(X_train_scaled)

    else:
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        train_pred = model.predict(X_train)

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    train_accuracy = accuracy_score(
        y_train,
        train_pred
    )

    results.append({
        "ML Model": name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MCC": mcc,
        "Training Accuracy": train_accuracy,
        "Testing Accuracy": accuracy
    })

    # Save model
    joblib.dump(
        model,
        f"model/{name}.pkl"
    )

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    print(f"Training Accuracy : {train_accuracy:.4f}")
    print(f"Testing Accuracy  : {accuracy:.4f}")
    print(f"AUC               : {auc:.4f}")
    print(f"Precision         : {precision:.4f}")
    print(f"Recall            : {recall:.4f}")
    print(f"F1 Score          : {f1:.4f}")
    print(f"MCC               : {mcc:.4f}")


# ---------------------------------------------------------
# 7. Save scaler
# ---------------------------------------------------------

joblib.dump(
    scaler,
    "model/scaler.pkl"
)


# ---------------------------------------------------------
# 8. Save test data
# ---------------------------------------------------------

test_data = X_test.copy()
test_data["diagnosis"] = y_test.values

test_data.to_csv(
    "test_data.csv",
    index=False
)


# ---------------------------------------------------------
# 9. Display final comparison table
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n\n")
print("=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)

print(
    results_df[
        [
            "ML Model",
            "Accuracy",
            "AUC",
            "Precision",
            "Recall",
            "F1",
            "MCC"
        ]
    ].round(4).to_string(index=False)
)

# ---------------------------------------------------------
# 10. Save model results
# ---------------------------------------------------------

results_df.to_csv(
    "model_results.csv",
    index=False
)


print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("model/logistic_regression.pkl")
print("model/decision_tree.pkl")
print("model/knn.pkl")
print("model/naive_bayes.pkl")
print("model/random_forest.pkl")
print("model/scaler.pkl")
print("test_data.csv")
print("model_results.csv")

print("\nTraining completed successfully!")

BREAST CANCER DATASET
X shape: (569, 30)
y shape: (569,)
Classes: [0, 1]

Training data: (455, 30)
Testing data : (114, 30)

LOGISTIC_REGRESSION
Training Accuracy : 0.9890
Testing Accuracy  : 0.9825
AUC               : 0.9954
Precision         : 0.9861
Recall            : 0.9861
F1 Score          : 0.9861
MCC               : 0.9623

DECISION_TREE
Training Accuracy : 1.0000
Testing Accuracy  : 0.9123
AUC               : 0.9157
Precision         : 0.9559
Recall            : 0.9028
F1 Score          : 0.9286
MCC               : 0.8174

KNN
Training Accuracy : 0.9736
Testing Accuracy  : 0.9561
AUC               : 0.9788
Precision         : 0.9589
Recall            : 0.9722
F1 Score          : 0.9655
MCC               : 0.9054

NAIVE_BAYES
Training Accuracy : 0.9407
Testing Accuracy  : 0.9386
AUC               : 0.9878
Precision         : 0.9452
Recall            : 0.9583
F1 Score          : 0.9517
MCC               : 0.8676

RANDOM_FOREST
Training Accuracy : 1.0000
Testing Accuracy  : 0.95